# Gold Layer, Daily Sales
**Source:** Silver Table `silver.silver_orders`  
**Target:** Gold Delta Table `/delta/gold/daily_sales` (Hive table `gold.gold_daily_sales`)  
**Pattern:** Full Refresh (`CREATE OR REPLACE`)  
**Transformations:**
- Group by `transaction_date`
- Calculate daily total sales (`SUM(total_amount)`)
  
**Layer:** Gold

---

In [3]:
import os, sys, logging
from pyspark.sql import SparkSession

os.environ["SPARK_HOME"] = "/opt/spark"
os.environ["PYTHONPATH"] = "/opt/spark/python:/opt/spark/python/lib/py4j-0.10.9.7-src.zip"
sys.path.insert(0, "/opt/spark/python")
sys.path.insert(0, "/opt/spark/python/lib/py4j-0.10.9.7-src.zip")

logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

CONFIG = {
    "silver_orders_db"   : "silver",
    "silver_orders_table": "silver_orders",
    "gold_db"            : "gold",
    "gold_table"         : "gold_daily_sales",
    "gold_path"          : "hdfs://master:8020/delta/gold/daily_sales",
}

def get_spark():
    return (
        SparkSession.builder
        .appName("Gold_Daily_Sales")
        .master("spark://master:7077")
        .config("spark.jars.packages",
                "io.delta:delta-spark_2.12:3.2.1")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
        .config("spark.hadoop.fs.defaultFS", "hdfs://master:8020")
        .config("spark.sql.warehouse.dir", "hdfs://master:8020/user/hive/warehouse")
        .config("spark.hadoop.hive.metastore.uris", "thrift://master:9083")
        .config("spark.databricks.delta.stats.collect", "false")
        .enableHiveSupport()
        .getOrCreate()
    )

def run():
    logger.info("Gold Daily Sales pipeline started.")
    spark = get_spark()
    spark.sparkContext.setLogLevel("ERROR")

    
    spark.sql(f"CREATE DATABASE IF NOT EXISTS {CONFIG['gold_db']}")

    # Build and execute the aggregation query
    query = f"""
        CREATE OR REPLACE TABLE {CONFIG['gold_db']}.{CONFIG['gold_table']}
        USING DELTA
        LOCATION '{CONFIG['gold_path']}'
        AS
        SELECT
            transaction_date,
            SUM(total_amount) AS daily_total_sales
        FROM {CONFIG['silver_orders_db']}.{CONFIG['silver_orders_table']}
        GROUP BY transaction_date
    """
    spark.sql(query)
    logger.info(f"Table {CONFIG['gold_db']}.{CONFIG['gold_table']} created/replaced.")

    # Verify and show sample
    count = spark.table(f"{CONFIG['gold_db']}.{CONFIG['gold_table']}").count()
    logger.info(f"SUCCESS: {count} rows in gold daily sales table.")
    spark.table(f"{CONFIG['gold_db']}.{CONFIG['gold_table']}").show(10, truncate=False)

if __name__ == "__main__":
    run()

2026-03-19 13:57:09,370 [INFO] Gold Daily Sales pipeline started.


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/jupyter/.ivy2/cache
The jars for the packages stored in: /home/jupyter/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-21b78c35-c02a-4b72-aeab-a9e0cfb82356;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.1 in central
	found io.delta#delta-storage;3.2.1 in central
	found org.antlr#antlr4-runtime;4.9.3 in central
downloading https://repo1.maven.org/maven2/io/delta/delta-spark_2.12/3.2.1/delta-spark_2.12-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-spark_2.12;3.2.1!delta-spark_2.12.jar (4533ms)
downloading https://repo1.maven.org/maven2/io/delta/delta-storage/3.2.1/delta-storage-3.2.1.jar ...
	[SUCCESSFUL ] io.delta#delta-storage;3.2.1!delta-storage.jar (147ms)
downloading https://repo1.maven.org/maven2/org/antlr/antlr4-runtime/4.9.3/antlr4-runtime-4.9.3.jar ...
	[SUCCESSFUL ] org.antlr#antlr4-runtime;4.9.3!antlr4-runtime.jar (404ms)
:: resolution report :: resolve 3691ms

+----------------+------------------+
|transaction_date|daily_total_sales |
+----------------+------------------+
|2021-01-27      |2907.0699999999997|
|2022-07-31      |3467.85           |
|2021-08-27      |1753.8000000000002|
|2022-03-28      |4544.17           |
|2021-11-13      |1589.25           |
|2020-08-24      |3680.6400000000003|
|2023-06-22      |3301.92           |
|2023-07-15      |4642.21           |
|2021-12-18      |4502.84           |
|2021-10-11      |1130.33           |
+----------------+------------------+
only showing top 10 rows

